# Stage 8: Results & Interpretation
Capstone Project: Spam Message Analysis Using Malicious URLs

Print the final numbers, interpret the findings, list limitations and next steps, and export the enriched dataset.

In [1]:
from pathlib import Path

# Project root = parent of this notebook's stage folder (run the notebook from inside its own folder)
ROOT = Path.cwd().parent
RAW_CSV = ROOT / "dataset" / "dataset.csv"
INTER = ROOT / "dataset" / "intermediate"      # output of each stage becomes input of the next
RESULTS = ROOT / "results"
GRAPH_DIR = RESULTS / "graphs"
for p in (INTER, RESULTS, GRAPH_DIR):
    p.mkdir(parents=True, exist_ok=True)


In [2]:
import pandas as pd
import json

df_final = pd.read_csv(INTER / "stage4_clean.csv", keep_default_na=False)
metrics = json.load(open(RESULTS / "metrics.json"))

print("Rows after cleaning:", len(df_final))
print("\nClass distribution (%):")
print((df_final["type"].value_counts(normalize=True) * 100).round(2))
print("\nHTTPS usage by class (%):")
print((df_final.groupby("type")["has_https"].mean() * 100).round(2))
print("\nAverage URL length by class:")
print(df_final.groupby("type")["url_length"].mean().round(1))
print("\nRandom Forest accuracy:", round(metrics["accuracy"] * 100, 2), "%")
print("Top 3 features:", metrics["top_features"])
print("Chi-square p-value (has_ip_address vs type):", metrics["chi2_p_value"])
print("\n", open(RESULTS / "classification_report.txt").read())


Rows after cleaning: 640088

Class distribution (%):
type
benign        66.77
defacement    14.87
phishing      14.68
malware        3.69
Name: proportion, dtype: float64

HTTPS usage by class (%):
type
benign         0.46
defacement     0.00
malware       28.43
phishing       7.40
Name: has_https, dtype: float64

Average URL length by class:
type
benign        57.7
defacement    86.2
malware       46.6
phishing      45.9
Name: url_length, dtype: float64

Random Forest accuracy: 88.47 %
Top 3 features: ['path_length', 'special_char_count', 'dot_count']
Chi-square p-value (has_ip_address vs type): 0.0

               precision    recall  f1-score   support

      benign       0.88      0.97      0.92     85477
  defacement       0.89      0.80      0.84     19030
     malware       0.99      0.77      0.87      4722
    phishing       0.85      0.62      0.72     18789

    accuracy                           0.88    128018
   macro avg       0.90      0.79      0.84    128018
weighted a

## Stage 8 — Results & Interpretation

The cell above prints the real numbers from Stages 4 to 6. Use them to confirm each finding below before the review.

**Key findings:**

- **Class balance:** the dataset is dominated by `benign` URLs (~66%), with `phishing`, `defacement`, and `malware` making up the rest — this imbalance matters when interpreting the classifier's accuracy.
- **HTTPS usage:** `has_https` differs sharply by class (see Chart 3) — malicious classes use HTTPS far less often than benign URLs, making it one of the strongest simple signals.
- **URL length:** `phishing`/`malware` URLs tend to run longer than `benign` ones (see Chart 2 boxplot), consistent with obfuscation and long redirect/query strings.
- **IP addresses in hostname:** the chi-square test shows a statistically significant relationship between `has_ip_address` and class — a classic phishing/malware red flag.
- **Suspicious keywords:** words like *login*, *verify*, *secure*, *account* cluster heavily in malicious URLs and rarely appear in benign ones.
- **Classifier performance:** the Random Forest accuracy and top features are printed in the cell above and saved in `results/metrics.json`.

**Limitations:**
- Purely lexical/structural features — no live network/WHOIS/content-based signals, so obfuscated or brand-new phishing domains without obvious lexical cues may be missed.
- Class imbalance can inflate overall accuracy; per-class precision/recall (Stage 6 classification report) is more informative than raw accuracy.

**Possible next steps:** add WHOIS/domain-age features, n-gram/TF-IDF text features on the URL string, or try a gradient-boosted model for comparison.

## Export enriched dataset

In [3]:
output_file = RESULTS / "malicious_url_enriched.csv"
df_final.to_csv(output_file, index=False)
print("Saved:", output_file)


Saved: C:\Users\pavan\Documents\Codex\2026-09-29\c-users-pavan-onedrive-documents-downloads\work\Spam-Message-Analysis-Using-Malicious-URLs\Spam-Message-Analysis-Using-Malicious-URLs\results\malicious_url_enriched.csv
